# Python程序设计（项目实战） · 工单P5《产线数据仓库与ORM》
**第7周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 复杂查询：多表JOIN+窗口函数（按小时聚合+环比）SQL与ORM双实现；示范讲解 |
| **B 组长** | 标准达成 | 数据字典完整+批量事务 |
| **C 操作员** | 基础完成 | 4表合理+ORM入库+2条查询 |
| **D 学徒** | 跟做保底 | 跟做示例跑通填空位1，程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

> **本notebook为完成态参考版**：学生版留空的每一格此处均已实现并**真实运行过**（输出已捕获）；
> 原填空处以 `# === 教师版完成态(原学生版填空位n: …) ===` 标记保留，课堂可直接指认要求。

## 一、课前资讯（Predict·预测）
> **预习快问**：一对多关系在表结构上如何表达？

> **课前检查**：模板仓库已克隆更新；导学单与实施计划已提交平台

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——把5个填空位（含4a/4b/4c）与A层拓展逐格补齐；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


## 三、任务要求（Make）
**实施步骤**
1. 四表模型（Device/SensorPoint/Record/AlarmEvent）+主外键+`relationship(back_populates)`+数据字典；
2. bulk批量事务入库（每批100条，一批一提交）；
3. 统计查询：各设备平均vib / 告警Top3设备 / 按小时聚合记录数；
4. 索引优化：`records(point_id, ts)` 联合索引，EXPLAIN QUERY PLAN + 前后耗时对比；
5. （A层）多表JOIN + 窗口函数（按小时聚合+环比），裸SQL与ORM双实现。

**验收标准**（工单《P5》「验收标准」四行原文）：`及格(D)`＝`4表设计合理；ORM入库成功；2条统计查询可用` ｜ `良好(C)`＝`数据字典完整；批量事务；溯源查询正确` ｜ `优秀(B)`＝`索引优化有EXPLAIN证据；N+1问题已规避` ｜ `超优(A)`＝`优秀+复杂查询(多表JOIN聚合)+导入P3仪表盘形成闭环演示`

> **口径说明（别把两套字母混为一谈）**：上四行是**工单《P5》「验收标准」原文**（及格/良好/优秀/超优），与本课课堂分层（D 跟做保底 / C 基础完成 / B 标准达成 / A 进阶拓展）**不同轴**——课堂 C≈工单及格、课堂 B≈工单良好，工单「优秀」档（如 索引优化有EXPLAIN证据；N+1问题已规避）课堂阶梯不覆盖。工程化项目的 `output/report.txt` 已按工单原文逐行取证：p11/p21/p33/p55 直接以这四行为分层口径，p12/p22/p41/p66 另设「工单验收四档对照」节；本机取不了证的子项逐条点名，不假判。

### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 本格学生版留空，此处为完成态）

In [ ]:
# -*- coding: utf-8 -*-
"""P5 产线数据仓库与ORM —— 完成态(SQLAlchemy + SQLite, 零安装/全离线/结果可复现)
数据: 由 random.seed(SEED) 合成 3设备×2测点 + 500条记录(约5%超阈值→生成告警)。
     想换成 P4 的真实采集数据(buffer.db)? 只改 init_and_seed() 里的记录来源即可,
     统计查询、窗口函数与索引实验都不受影响。
"""
import contextlib
import random
import sqlite3
from datetime import datetime, timedelta

import sqlalchemy
from sqlalchemy import (Boolean, Column, DateTime, Float, ForeignKey, Index,
                        Integer, String, create_engine, func, join, text)
from sqlalchemy.orm import Session as _RawSession
from sqlalchemy.orm import relationship, selectinload

# ---- 版本兼容层: 本notebook按 SQLAlchemy 2.x 写法编写(requirements.lock=2.0.54) ----
# 若机器上还是 1.3/1.4 老版本, 下面几行自动降级适配, 单元格里
# select(列, …) / with Session(e) as s / session.execute(…).fetchall() 一行都不用改。
SA_VER = tuple(int(x) for x in sqlalchemy.__version__.split('.')[:2])
if SA_VER < (1, 4):
    from sqlalchemy import select as _select_legacy

    def select(*args, **kw):              # 1.3 要求 select([列, ...])
        return _select_legacy(list(args), **kw)
else:
    from sqlalchemy import select         # 2.x 原生 select(列, ...)

if hasattr(_RawSession, '__enter__'):
    Session = _RawSession                 # 2.x: with Session(engine) as s 原生可用
else:                                     # 1.3 的 Session 不支持 with, 包一层等价语义
    @contextlib.contextmanager
    def Session(engine):
        s = _RawSession(engine)
        try:
            yield s
        finally:
            s.close()

DB = "sqlite:///warehouse.db"         # cell2 已 os.chdir 到课程包根目录 → 库落在根目录
SEED = 20260401                       # 固定随机种子: 每次跑出的数字完全一致
N_RECORDS = 500                       # 采集记录条数(工单要求)
BATCH = 100                           # 批量事务: 每100条一批一提交
SPIKE_RATE = 0.05                     # 约5%记录超过测点阈值 → 触发告警
STEP_MIN = 3                          # 记录间隔3分钟 → 500条跨约25小时, 小时桶铺开
BASE_TS = datetime(2026, 4, 1, 8, 0)

print(f"[env] 工作目录={os.getcwd()}  SQLAlchemy={sqlalchemy.__version__} "
      f"SQLite={sqlite3.sqlite_version}(窗口函数需≥3.25)")

### 💡 分层提示（先自己试5分钟再看）
**思路提示（按填空位逐条给）**
> - **填空位1 · Device**：加一列 `area = Column(String(32), nullable=False, default="未分配车间")`；字段名/类型/含义同时写进类docstring里的数据字典。
> - **填空位2 · Record**：类里加 `__table_args__ = (Index("idx_dev_ts", "point_id", "ts"),)`；这条联合索引服务"按测点查时段"这一最高频查询，注意最左前缀（只按 `ts` 过滤用不上它）。
> - **填空位3 · init_and_seed()**：`random.seed(SEED)` → 3设备×各2测点用 `add_all` → 500条记录用 `bulk_save_objects` 每100条一批并 `commit`（批量事务）→ vib 约5% > 4.5 时生成告警并回填 `record_id` 便于溯源。
> - **填空位4a · 各设备平均vib**：`Device → SensorPoint → Record` 两级JOIN，`where(SensorPoint.metric == "vib")` + `func.avg(Record.value)` + `group_by(Device.code)`。
> - **填空位4b · 告警Top3设备**：`Device JOIN AlarmEvent` + `func.count` + `group_by` + `order_by(...desc()).limit(3)`。
> - **填空位4c · 按小时聚合**：SQLite 没有 `date_trunc`，改用 `func.strftime("%Y-%m-%d %H", Record.ts)` 分组；各小时计数之和应等于 500。
> - **填空位5 · optimize()**：先删掉建表时自动创建的 `idx_dev_ts` 保留"索引前"现场 → `EXPLAIN QUERY PLAN`（应见 SCAN 全表）+ 计时 → `CREATE INDEX` → 再EXPLAIN（应见 SEARCH ... USING INDEX）+ 再计时，两个 ms 写进工单。
> - **A层拓展 · 窗口函数**：`ROW_NUMBER()/LAG() OVER (PARTITION BY dev ORDER BY hour)` 实现"按小时聚合+环比"，裸SQL 与 ORM（`func.row_number()/func.lag()`）各写一遍，最后 `assert` 两种实现逐行一致。

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位填空位行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改批次大小/换索引列对比效果，A层再补窗口函数双实现，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方“验收标准”四行逐条自查并在工单上签字，未做到的档位如实写「未做」——不得只勾 及格(D) 与 优秀(B+)：`及格(D)`＝`4表设计合理；ORM入库成功；2条统计查询可用` ｜ `良好(C)`＝`数据字典完整；批量事务；溯源查询正确` ｜ `优秀(B)`＝`索引优化有EXPLAIN证据；N+1问题已规避` ｜ `超优(A)`＝`优秀+复杂查询(多表JOIN聚合)+导入P3仪表盘形成闭环演示`

### Starter 2：`Base = make_base()`（2.x即 class Base(DeclarativeBase)）
（Investigate: 读代码→Modify: 本格学生版留空，此处为完成态）

In [ ]:
def make_base():
    """给出一个全新的声明式基类: 2.x 用 DeclarativeBase, 老环境自动回退 declarative_base()"""
    try:
        from sqlalchemy.orm import DeclarativeBase          # SQLAlchemy 2.x 写法

        class _Base(DeclarativeBase):
            pass
    except ImportError:                                     # 1.3/1.4 环境
        from sqlalchemy.ext.declarative import declarative_base
        _Base = declarative_base()
    return _Base


Base = make_base()

### Starter 3：`class Device(Base):`
（Investigate: 读代码→Modify: 本格学生版留空，此处为完成态）

In [ ]:
class Device(Base):
    """设备表(档案主表)
    数据字典: id INTEGER 主键自增 | code VARCHAR(16) 设备编号(唯一)
              name VARCHAR(64) 设备名称 | area VARCHAR(32) 所属车间
    """
    __tablename__ = "devices"
    id = Column(Integer, primary_key=True)
    code = Column(String(16), unique=True, nullable=False)
    name = Column(String(64))
    points = relationship("SensorPoint", back_populates="device")
    alarms = relationship("AlarmEvent", back_populates="device")

    # === 教师版完成态(原学生版填空位1: 补充 area(车间) 字段) ===
    area = Column(String(32), nullable=False, default="未分配车间")

    def __repr__(self):
        return f"<Device {self.code} {self.name} @{self.area}>"

### Starter 4：`class SensorPoint(Base):`
（Investigate: 读代码→Modify: 本格学生版留空，此处为完成态）

In [ ]:
class SensorPoint(Base):
    """测点表(设备↔测点 一对多)
    数据字典: id INTEGER 主键 | device_id INT FK→devices.id(外键挡住孤儿记录)
              metric VARCHAR(16) 量类型 vib/temp/press | unit VARCHAR(8) 单位
              label VARCHAR(32) 测点名称 | warn_limit FLOAT 告警阈值(vib=4.5)
    """
    __tablename__ = "points"
    id = Column(Integer, primary_key=True)
    device_id = Column(Integer, ForeignKey("devices.id"), nullable=False)
    metric = Column(String(16))          # temp / vib / press
    unit = Column(String(8))
    device = relationship("Device", back_populates="points")

    # === 教师版完成态(4表模型补全: 测点名称/告警阈值 + 与记录、告警的双向关系) ===
    label = Column(String(32))
    warn_limit = Column(Float, default=4.5)
    records = relationship("Record", back_populates="point")
    alarms = relationship("AlarmEvent", back_populates="point")

    def __repr__(self):
        return f"<Point {self.label or self.id} {self.metric}({self.unit})>"

### Starter 5：`class Record(Base):`
（Investigate: 读代码→Modify: 本格学生版留空，此处为完成态）

In [ ]:
class Record(Base):
    """采集记录表(高频写入, 索引优化重点)
    数据字典: id INTEGER 主键 | point_id INT FK→points.id
              ts DATETIME 采集时刻(DateTime, 存ISO勿存字符串) | value FLOAT 测量值
    """
    __tablename__ = "records"
    id = Column(Integer, primary_key=True)
    point_id = Column(Integer, ForeignKey("points.id"), nullable=False)
    ts = Column(DateTime, default=datetime.utcnow)
    value = Column(Float)
    point = relationship("SensorPoint", back_populates="records")

    # === 教师版完成态(原学生版填空位2: 联合索引 Index('idx_dev_ts', point_id, ts)) ===
    # 最左前缀: WHERE point_id / point_id+ts 才命中; 只给 ts 加条件用不上。
    # records 写多读少, 索引有写代价 → 只保留这一条联合索引, ts 不再单独建。
    __table_args__ = (Index("idx_dev_ts", "point_id", "ts"),)
    raised_alarm = relationship("AlarmEvent", back_populates="source_record",
                                uselist=False)

### Starter 6：`class AlarmEvent(Base):`
（Investigate: 读代码→Modify: 本格学生版留空，此处为完成态）

In [ ]:
class AlarmEvent(Base):
    """告警事件表(挂设备+挂测点, 并溯源到触发它的原始记录)
    数据字典: id INTEGER 主键 | device_id INT FK→devices.id
              point_id INT FK→points.id | record_id INT FK→records.id(溯源)
              ts DATETIME 告警时刻 | level INT 2=预警/3=停机
              message VARCHAR(128) 描述 | resolved BOOLEAN 是否闭环
    """
    __tablename__ = "alarms"
    id = Column(Integer, primary_key=True)
    device_id = Column(Integer, ForeignKey("devices.id"), nullable=False)
    ts = Column(DateTime, default=datetime.utcnow)
    level = Column(Integer)
    message = Column(String(128))
    resolved = Column(Boolean, default=False)

    # === 教师版完成态(4表模型补全: 测点/记录外键 + 三个relationship + __repr__) ===
    point_id = Column(Integer, ForeignKey("points.id"))
    record_id = Column(Integer, ForeignKey("records.id"))
    device = relationship("Device", back_populates="alarms")
    point = relationship("SensorPoint", back_populates="alarms")
    source_record = relationship("Record", back_populates="raised_alarm")

    def __repr__(self):
        return f"<Alarm dev={self.device_id} L{self.level} {self.ts:%m-%d %H:%M}>"

### Starter 7：`def init_and_seed():`
（Investigate: 读代码→Modify: 本格学生版留空，此处为完成态）

In [ ]:
def init_and_seed(batch=BATCH):
    """建库 → 建表 → 批量事务灌种子数据(drop_all→create_all, 可反复重跑)"""
    e = create_engine(DB, echo=False)
    Base.metadata.drop_all(e)
    Base.metadata.create_all(e)

    # __table_args__ 里的 idx_dev_ts 会被 create_all 直接建出来;
    # 为保留"索引前"现场(填空位5要做前后对比), 这里先删掉, 由 optimize() 建回来
    with e.begin() as c:                      # with e.begin(): 正常退出即提交(2.x推荐写法)
        c.execute(text("DROP INDEX IF EXISTS idx_dev_ts"))

    random.seed(SEED)                         # 固定种子 → 每次都是同一批500条数据
    with Session(e) as s:
        # === 教师版完成态(原学生版填空位3: 3设备×各2测点 + 模拟记录500条) ===
        s.add_all([Device(code=f"DEV-{i:02d}", name=n, area=a)
                   for i, (n, a) in enumerate([("空压机", "冲压车间"),
                                               ("冷却泵", "焊接车间"),
                                               ("传送带", "总装车间")], start=1)])
        s.flush()
        # 取回整个实体: session.query(…) 在 2.x 仍可用(等价 select(Device)+scalars())
        devs = s.query(Device).order_by(Device.id).all()

        pts = []
        for d in devs:
            pts.append(SensorPoint(device_id=d.id, metric="vib", unit="mm/s",
                                   label=f"{d.code}-振动", warn_limit=4.5))
            pts.append(SensorPoint(device_id=d.id, metric="temp", unit="degC",
                                   label=f"{d.code}-温度", warn_limit=80.0))
        s.add_all(pts)
        s.flush()

        rows, spikes, rid = [], [], 0
        for i in range(N_RECORDS):
            p = pts[i % len(pts)]                         # 6个测点轮转采样
            ts = BASE_TS + timedelta(minutes=STEP_MIN * i)
            over = random.random() < SPIKE_RATE           # 约5%记录超阈值
            if p.metric == "vib":
                val = random.uniform(4.55, 6.4) if over else random.uniform(1.0, 4.4)
            else:
                val = random.uniform(80.5, 88.0) if over else random.uniform(45.0, 79.0)
            val = round(val, 3)
            rid += 1
            rows.append(Record(id=rid, point_id=p.id, ts=ts, value=val))
            if val > p.warn_limit:
                spikes.append((rid, p, ts, val))

        # B层要求"批量事务": bulk_save_objects 每100条一批, 一批一次 commit
        n_batch = 0
        for k in range(0, len(rows), batch):
            s.bulk_save_objects(rows[k:k + batch])
            s.commit()
            n_batch += 1

        # 超阈值记录 → 告警事件(record_id 让告警能反查原始记录 = 故障溯源)
        s.add_all([AlarmEvent(device_id=p.device_id, point_id=p.id, record_id=r_id,
                              ts=ts, level=3 if val > p.warn_limit + 1.0 else 2,
                              message=f"{p.metric}={val} 超阈值{p.warn_limit}{p.unit}",
                              resolved=False)
                   for r_id, p, ts, val in spikes])
        s.commit()

        counts = {t.__tablename__: s.execute(select(func.count()).select_from(t))
                  .scalar() for t in (Device, SensorPoint, Record, AlarmEvent)}
        print(f"[init] 库={DB}")
        print(f"[init] 批量事务: {len(rows)}条记录 / 每批{batch}条 → {n_batch}次commit")
        print(f"[init] 行数: 设备{counts['devices']} 测点{counts['points']} "
              f"记录{counts['records']} 告警{counts['alarms']}"
              f"(告警率{counts['alarms'] / counts['records']:.1%})")
        assert counts["records"] == N_RECORDS, counts
    return e

### Starter 8：`def queries(e):`
（Investigate: 读代码→Modify: 本格学生版留空，此处为完成态）

In [ ]:
def queries(e):
    """完成态: 三条统计查询(多表JOIN + GROUP BY + 聚合) + N+1规避演示"""
    with Session(e) as s:
        # === 教师版完成态(原学生版填空位4a: 各设备平均vib — JOIN+GROUP BY → func.avg) ===
        avg_vib = func.round(func.avg(Record.value), 3)
        frm = (join(Device, SensorPoint, SensorPoint.device_id == Device.id)
               .join(Record, Record.point_id == SensorPoint.id))
        stmt = (select(Device.code, Device.area, func.count(Record.id), avg_vib)
                .select_from(frm)
                .where(SensorPoint.metric == "vib")
                .group_by(Device.code, Device.area)
                .order_by(avg_vib.desc()))
        print("4a 各设备平均vib(mm/s):")
        for code_, area, n, av in s.execute(stmt).fetchall():
            print(f"   {code_} | {area} | vib记录{n}条 | 平均 {av}")

        # === 教师版完成态(原学生版填空位4b: 告警Top3设备) ===
        n_alarm = func.count(AlarmEvent.id)
        stmt = (select(Device.code, Device.name, n_alarm, func.max(AlarmEvent.level))
                .select_from(join(Device, AlarmEvent,
                                  AlarmEvent.device_id == Device.id))
                .group_by(Device.code, Device.name)
                .order_by(n_alarm.desc(), Device.code)
                .limit(3))
        print("4b 告警Top3设备:")
        for i, (code_, name, n, lv) in enumerate(s.execute(stmt).fetchall(), start=1):
            print(f"   Top{i} {code_} {name}: 告警{n}条 (最高级别 L{lv})")

        # === 教师版完成态(原学生版填空位4c: 按小时聚合记录数) ===
        # SQLite 没有 date_trunc, 用 strftime; 格式串取到小时 = 一个整点桶
        hour = func.strftime("%Y-%m-%d %H", Record.ts).label("hour")
        stmt = (select(hour, func.count(Record.id))
                .group_by(hour).order_by(hour))
        buckets = s.execute(stmt).fetchall()
        print("4c 按小时聚合记录数:")
        for h, n in buckets[:5]:
            print(f"   {h} 时 {n}条")
        total = sum(n for _, n in buckets)
        peak = max(buckets, key=lambda r: r[1])
        print(f"   …共{len(buckets)}个整点桶, 峰值{peak[0]}({peak[1]}条), "
              f"合计{total}条 (应等于{N_RECORDS})")
        assert total == N_RECORDS, total

        # 附带演示(B级"N+1已规避"): selectinload 预加载, 共3条SQL而不是1+3×2条
        #   两个都是集合关系, 用 joinedload 会放大出重复行, 故这里选 selectinload
        print("附: 设备→测点/告警 关系遍历(selectinload 预加载):")
        for d in (s.query(Device)
                  .options(selectinload(Device.points), selectinload(Device.alarms))
                  .order_by(Device.code).all()):
            first = d.alarms[0] if d.alarms else None
            print(f"   {d!r} → 测点{len(d.points)}个 告警{len(d.alarms)}条 首个告警 {first!r}")

### Starter 9：`def optimize(e):`
（Investigate: 读代码→Modify: 本格学生版留空，此处为完成态）

In [ ]:
def optimize(e, iters=400):
    """填空位5: 加索引前后查询耗时对比 + EXPLAIN QUERY PLAN 证据(用 text() 走原生SQL)"""
    import time
    # 高频查询①(明细): 某测点某时段的记录按时间排序 —— KPI曲线/故障溯源都长这样
    detail = ("SELECT id, point_id, ts, value FROM records "
              "WHERE point_id = :pid AND ts BETWEEN :t0 AND :t1 ORDER BY ts")
    # 高频查询②(计数): 同样的过滤条件只问"有几条" —— 只碰索引就能答(覆盖索引)
    counting = ("SELECT COUNT(*) FROM records "
                "WHERE point_id = :pid AND ts BETWEEN :t0 AND :t1")
    # DateTime 列底层就是 ISO 文本, 原生SQL传字符串即可(比传 datetime 对象更稳)
    params = {"pid": 2,
              "t0": f"{BASE_TS + timedelta(hours=2):%Y-%m-%d %H:%M:%S}",
              "t1": f"{BASE_TS + timedelta(hours=10):%Y-%m-%d %H:%M:%S}"}
    # 同一条查询的ORM写法(课堂对照): select(Record).where(Record.point_id == 2,
    #   Record.ts.between(t0, t1)).order_by(Record.ts)

    def measure(tag, sql):
        with e.connect() as c:
            for row in c.execute(text("EXPLAIN QUERY PLAN " + sql), params).fetchall():
                print(f"        [{tag}] {row[3]}")
            c.execute(text(sql), params).fetchall()        # 预热, 排除冷启动建计划的噪声
            t0 = time.perf_counter()
            for _ in range(iters):
                rows = c.execute(text(sql), params).fetchall()
            ms = (time.perf_counter() - t0) * 1000
            print(f"[{tag}] {iters}次 {ms:.2f}ms (单次 {ms / iters:.4f}ms), 返回{len(rows)}行")
            return ms, len(rows)

    b_detail = measure("索引前 无idx_dev_ts", detail)
    b_count = measure("索引前 无idx_dev_ts", counting)
    with e.begin() as c:            # 建索引: 与 __table_args__ 里声明的同名同列
        c.execute(text("CREATE INDEX IF NOT EXISTS idx_dev_ts ON records(point_id, ts)"))
    a_detail = measure("索引后 有idx_dev_ts", detail)
    a_count = measure("索引后 有idx_dev_ts", counting)
    assert b_detail[1] == a_detail[1] and b_count[1] == a_count[1], "索引不应改变结果集"

    print(f"结论: 明细查询 {b_detail[0]:.1f}ms → {a_detail[0]:.1f}ms, 提速 "
          f"{b_detail[0] / a_detail[0]:.1f}x (单次 {b_detail[0] / iters:.4f}ms → "
          f"{a_detail[0] / iters:.4f}ms)")
    print(f"      计数查询 {b_count[0]:.1f}ms → {a_count[0]:.1f}ms, 提速 "
          f"{b_count[0] / a_count[0]:.1f}x —— 两个数字都填进工单'参数-收益'栏")
    print("计划变化: SCAN records(+USE TEMP B-TREE FOR ORDER BY) → "
          "SEARCH records USING [COVERING] INDEX idx_dev_ts (point_id=? AND ts>? AND ts<?)")
    print("代价提醒: 500行小表的差距会被每次调用的固定开销稀释, 真实产线百万行时按数量级放大; "
          "索引占盘且拖慢写入, 写多读少的表慎滥建(B06: 最左前缀/覆盖索引)")

### 拓展（A层·师傅）：`def window_queries(e):`
> 工单 **A层（进阶拓展）**：复杂查询 = **多表JOIN + 窗口函数（按小时聚合 + 环比）**，且 **SQL 与 ORM 双实现**。
> 本格学生版留空，教师版给完成态：先 `JOIN records→points→devices` 按 `dev×hour` 聚合，
> 再用 `ROW_NUMBER() OVER (PARTITION BY dev ORDER BY hour)` 排名、`LAG(cnt)` 取上一小时条数算环比；
> ORM 侧用 `func.row_number()/func.lag()` + `over(partition_by=…, order_by=…)` 写同一条查询，
> 最后逐行 `assert` 两种实现结果一致（SQLite 3.25+ 起支持窗口函数，本机 3.42，无需额外依赖）。

In [ ]:
def window_queries(e, show=8):
    """A层完成态: 多表JOIN + 窗口函数(按小时聚合+环比) —— 裸SQL 与 ORM 双实现
    环比 mom_pct = (本小时条数 - 上一小时条数) / 上一小时条数 × 100%
    """
    hour = func.strftime("%Y-%m-%d %H", Record.ts).label("hour")

    def line(dev, hr, cnt, rn, prev, mom):
        rate = "—" if mom is None else f"{mom:+.1f}%"
        return f"   {dev} | {hr} | 记录{cnt:>2}条 | rn={rn:<2}| 上小时={str(prev):>4} | 环比={rate}"

    # ---------- 实现1: 原生SQL —— CTE 聚合 + ROW_NUMBER()/LAG() OVER(...) ----------
    sql = """
    WITH h AS (
      SELECT d.code AS dev, strftime('%Y-%m-%d %H', r.ts) AS hour, COUNT(r.id) AS cnt
      FROM records r
      JOIN points  p ON r.point_id  = p.id
      JOIN devices d ON p.device_id = d.id
      GROUP BY dev, hour
    )
    SELECT dev, hour, cnt,
           ROW_NUMBER() OVER (PARTITION BY dev ORDER BY hour) AS rn,
           LAG(cnt)     OVER (PARTITION BY dev ORDER BY hour) AS prev_cnt,
           ROUND((cnt - LAG(cnt) OVER (PARTITION BY dev ORDER BY hour)) * 100.0
                 / NULLIF(LAG(cnt) OVER (PARTITION BY dev ORDER BY hour), 0), 1) AS mom_pct
    FROM h
    ORDER BY dev, hour
    """
    with Session(e) as s:
        raw = [tuple(r) for r in s.execute(text(sql)).fetchall()]

        # ---------- 实现2: ORM —— 同一条查询用 func.row_number()/func.lag() 表达 ----------
        # (窗口函数在 GROUP BY 之后计算, 所以聚合与窗口可以写在同一层select里)
        frm = (join(Record, SensorPoint, Record.point_id == SensorPoint.id)
               .join(Device, SensorPoint.device_id == Device.id))
        lag_cnt = func.lag(func.count(Record.id)).over(partition_by=Device.code,
                                                       order_by=hour)
        stmt = (select(Device.code, hour, func.count(Record.id),
                       func.row_number().over(partition_by=Device.code,
                                              order_by=hour).label("rn"),
                       lag_cnt.label("prev_cnt"),
                       func.round((func.count(Record.id) - lag_cnt) * 100.0
                                  / func.nullif(lag_cnt, 0), 1).label("mom_pct"))
                .select_from(frm)
                .group_by(Device.code, hour)
                .order_by(Device.code, hour))
        orm = [tuple(r) for r in s.execute(stmt).fetchall()]

    print(f"A层·实现1 原生SQL窗口函数 (CTE+ROW_NUMBER/LAG, 共{len(raw)}行, 前{show}行):")
    for row in raw[:show]:
        print(line(*row))
    print(f"A层·实现2 ORM窗口函数 (func.row_number/func.lag, 共{len(orm)}行, 前{show}行):")
    for row in orm[:show]:
        print(line(*row))

    # ---------- 双实现互校 + 总量守恒 ----------
    assert raw == orm, "SQL与ORM双实现必须逐行一致"
    assert sum(r[2] for r in orm) == N_RECORDS, sum(r[2] for r in orm)
    print(f"   ✓ 两种实现逐行一致；设备×小时共{len(orm)}行, 记录数合计"
          f"{sum(r[2] for r in orm)}条 = {N_RECORDS}")

### Starter 10：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 本格学生版留空，此处为完成态）

In [ ]:
if __name__ == "__main__":
    e = init_and_seed()      # 建库建表 + 批量事务灌种子(drop_all→create_all, 可重跑)
    queries(e)               # C/B层: 三条统计查询 + N+1规避
    window_queries(e)        # A层: 多表JOIN + 窗口函数(SQL/ORM 双实现)
    optimize(e)              # B层: 索引前后耗时 + EXPLAIN 证据
    print("\n[P5 完成态] 4表模型 + 批量事务 + 统计查询 + 窗口函数 + 索引实验 全部跑通")

## 四、提交要求与避坑
- 提交: `学号_姓名_P5.py` 或本notebook(含输出) + ER图 + 数据字典.md + 优化前后耗时对比
- ⚠ DateTime存ISO，别存字符串
- ⚠ bulk_save_objects批量插入比逐条add快10倍+，**每批必须commit**，否则进程一退全没
- ⚠ session用完即with关闭，防连接泄漏
- ⚠ relationship 缺 back_populates 会重复查询；循环里访问 `d.points` 触发 N+1 → `selectinload`/`joinedload` 预加载
- ⚠ 索引不是越多越好：records 写多读少，只留 `(point_id, ts)` 一条联合索引
- ⚠ 本机 SQLAlchemy 若是 1.3/1.4，Starter 1 的版本兼容层会自动降级，代码无需改动

---

## 教师参考答案（来源：`教师参考/solutions.py` · `p55()`）

> **发放纪律：本附录勿随学生版下发。** 课堂按层放提示（D层演示照做/C层只给思路行/A层直接做）。

学生版留空格（任务点1…5，含 4a/4b/4c 与 A层拓展）→ 本notebook完成态 → `p55()` 参考实现的对应关系：

| 学生版留空格 | 原要求（一句话） | 本notebook完成态 | 关键实现要点 | 达标层 |
|---|---|---|---|---|
| 填空位1 `Device` | 补充 area(车间) 字段 | Starter 3 `class Device` | `Column(String(32), nullable=False, default=…)` + docstring数据字典 + `__repr__` | D |
| 填空位2 `Record` | 联合索引 `Index('idx_dev_ts', point_id, ts)` | Starter 5 `class Record` | `__table_args__ = (Index(…),)`；最左前缀；只留一条索引控写代价 | B |
| 填空位3 `init_and_seed()` | 3设备×2测点 + 500条记录(5%超阈值告警) | Starter 7 | `random.seed(SEED)` + `add_all` + `bulk_save_objects` 每100条一commit + 告警回填record_id | C/B |
| 填空位4a `queries()` | 各设备平均vib（JOIN+GROUP BY→func.avg） | Starter 8 | `join()` 两级JOIN + `where(metric=='vib')` + `func.avg` + `group_by` | C |
| 填空位4b `queries()` | 告警Top3设备 | Starter 8 | `Device JOIN AlarmEvent` + `func.count` + `order_by(desc).limit(3)` | C |
| 填空位4c `queries()` | 按小时聚合记录数（`date_trunc`不可用→`strftime`） | Starter 8 | `func.strftime('%Y-%m-%d %H', ts)` 分组，各桶之和=500 | B |
| 填空位5 `optimize()` | 加索引前后耗时对比 + `EXPLAIN QUERY PLAN` | Starter 9 | 先删索引测SCAN → `CREATE INDEX` → 测SEARCH，`text()` 取计划行 | B |
| A层拓展 `window_queries()` | 多表JOIN+窗口函数(小时聚合+环比)，SQL与ORM双实现 | 拓展格（Starter 9 之后） | `ROW_NUMBER()/LAG() OVER(PARTITION BY dev ORDER BY hour)` 与 `func.row_number()/func.lag()`，逐行 `assert` 一致 | A |

> 数据字典与4表补全（`SensorPoint.label/warn_limit`、`AlarmEvent.point_id/record_id` 溯源外键、
> 三组 `relationship`）散落在 Starter 3/4/5/6 四个完成态格里，满足B层"数据字典完整"；
> 下方 `p55()` 是等价的独立可运行版（自建4表 + 断言行数），与上方完成态互为校验。

In [ ]:
def p55():
    """教师参考答案: P5 完整可运行版 —— 4表+数据字典+批量事务+统计查询+索引前后对比
    独立建库 ref_warehouse.db(不动 warehouse.db), 复用本notebook的 select/Session/make_base,
    与上方完成态互为校验; 关键处都有行数断言, 跑通即证明数字是真的。
    """
    import random
    import time

    from sqlalchemy import (Boolean, Column, DateTime, Float, ForeignKey, Index,
                            Integer, String, create_engine, func, join, text)

    RBase = make_base()

    class RDev(RBase):                        # devices: 设备档案(填空位1 的 area)
        __tablename__ = "devices"
        id = Column(Integer, primary_key=True)
        code = Column(String(16), unique=True, nullable=False)
        name = Column(String(64))
        area = Column(String(32), nullable=False, default="未分配车间")

    class RPoint(RBase):                      # points: 测点(阈值供告警用)
        __tablename__ = "points"
        id = Column(Integer, primary_key=True)
        device_id = Column(Integer, ForeignKey("devices.id"), nullable=False)
        metric = Column(String(16))
        unit = Column(String(8))
        warn_limit = Column(Float, default=4.5)

    class RRec(RBase):                        # records: 采集记录(填空位2 的联合索引)
        __tablename__ = "records"
        id = Column(Integer, primary_key=True)
        point_id = Column(Integer, ForeignKey("points.id"), nullable=False)
        ts = Column(DateTime, default=datetime.utcnow)
        value = Column(Float)
        __table_args__ = (Index("idx_dev_ts", "point_id", "ts"),)

    class RAlarm(RBase):                      # alarms: 告警(record_id 可溯源)
        __tablename__ = "alarms"
        id = Column(Integer, primary_key=True)
        device_id = Column(Integer, ForeignKey("devices.id"), nullable=False)
        point_id = Column(Integer, ForeignKey("points.id"))
        record_id = Column(Integer, ForeignKey("records.id"))
        ts = Column(DateTime, default=datetime.utcnow)
        level = Column(Integer)
        message = Column(String(128))
        resolved = Column(Boolean, default=False)

    e = create_engine("sqlite:///ref_warehouse.db", echo=False)
    RBase.metadata.drop_all(e)
    RBase.metadata.create_all(e)
    with e.begin() as c:                       # 留"索引前"现场
        c.execute(text("DROP INDEX IF EXISTS idx_dev_ts"))

    random.seed(7)                             # 参考答案自己的种子, 数字同样可复现
    base = datetime(2026, 4, 1, 8, 0)
    with Session(e) as s:
        s.add_all([RDev(code=f"DEV-{i:02d}", name=n, area=a)
                   for i, (n, a) in enumerate([("空压机", "冲压车间"),
                                               ("冷却泵", "焊接车间"),
                                               ("传送带", "总装车间")], start=1)])
        s.flush()
        devs = s.query(RDev).order_by(RDev.id).all()
        pts = []
        for d in devs:
            pts.append(RPoint(device_id=d.id, metric="vib", unit="mm/s", warn_limit=4.5))
            pts.append(RPoint(device_id=d.id, metric="temp", unit="degC", warn_limit=80.0))
        s.add_all(pts)
        s.flush()

        recs, spikes = [], []
        for i in range(500):
            p = pts[i % len(pts)]
            ts = base + timedelta(minutes=3 * i)
            over = random.random() < 0.05
            if p.metric == "vib":
                v = random.uniform(4.55, 6.4) if over else random.uniform(1.0, 4.4)
            else:
                v = random.uniform(80.5, 88.0) if over else random.uniform(45.0, 79.0)
            v = round(v, 3)
            recs.append(RRec(id=i + 1, point_id=p.id, ts=ts, value=v))
            if v > p.warn_limit:
                spikes.append((i + 1, p, ts, v))
        for k in range(0, len(recs), 100):     # 批量事务: 每100条一commit
            s.bulk_save_objects(recs[k:k + 100])
            s.commit()
        s.add_all([RAlarm(device_id=p.device_id, point_id=p.id, record_id=r_id, ts=ts,
                          level=3 if v > p.warn_limit + 1 else 2,
                          message=f"{p.metric}={v} 超阈值{p.warn_limit}")
                   for r_id, p, ts, v in spikes])
        s.commit()

        n_dev = s.execute(select(func.count()).select_from(RDev)).scalar()
        n_pt = s.execute(select(func.count()).select_from(RPoint)).scalar()
        n_rc = s.execute(select(func.count()).select_from(RRec)).scalar()
        n_al = s.execute(select(func.count()).select_from(RAlarm)).scalar()
        print(f"[p55] 4表建库+种子: 设备{n_dev} 测点{n_pt} 记录{n_rc} 告警{n_al}")
        assert (n_dev, n_pt, n_rc) == (3, 6, 500), (n_dev, n_pt, n_rc)
        assert n_al > 0 and abs(n_al / n_rc - 0.05) < 0.04, n_al

        avg = func.round(func.avg(RRec.value), 3)
        frm = (join(RDev, RPoint, RPoint.device_id == RDev.id)
               .join(RRec, RRec.point_id == RPoint.id))
        print("[p55] 按设备平均振动 vib(mm/s):")
        for code_, area, cnt, av in s.execute(
                select(RDev.code, RDev.area, func.count(RRec.id), avg)
                .select_from(frm).where(RPoint.metric == "vib")
                .group_by(RDev.code, RDev.area).order_by(avg.desc())).fetchall():
            print(f"        {code_} {area} {cnt}条 平均{av}")

        n_a = func.count(RAlarm.id)
        print("[p55] 告警Top3设备:")
        for i, (code_, k) in enumerate(s.execute(
                select(RDev.code, n_a)
                .select_from(join(RDev, RAlarm, RAlarm.device_id == RDev.id))
                .group_by(RDev.code).order_by(n_a.desc(), RDev.code)
                .limit(3)).fetchall(), start=1):
            print(f"        Top{i} {code_}: {k}条")

        hr = func.strftime("%Y-%m-%d %H", RRec.ts).label("hour")
        hourly = s.execute(select(hr, func.count(RRec.id))
                           .group_by(hr).order_by(hr)).fetchall()
        print(f"[p55] 按小时聚合: {len(hourly)}个整点桶, 前3={hourly[:3]}, "
              f"合计={sum(n for _, n in hourly)}")
        assert sum(n for _, n in hourly) == 500

    probe = ("SELECT id, point_id, ts, value FROM records "
             "WHERE point_id = :pid AND ts BETWEEN :t0 AND :t1 ORDER BY ts")
    prm = {"pid": 2, "t0": f"{base + timedelta(hours=2):%Y-%m-%d %H:%M:%S}",
           "t1": f"{base + timedelta(hours=10):%Y-%m-%d %H:%M:%S}"}

    def timed(tag, iters=400):
        with e.connect() as c:
            plan = [r[3] for r in c.execute(text("EXPLAIN QUERY PLAN " + probe), prm)]
            c.execute(text(probe), prm).fetchall()
            t0 = time.perf_counter()
            for _ in range(iters):
                rows = c.execute(text(probe), prm).fetchall()
            ms = (time.perf_counter() - t0) * 1000
            print(f"[p55] {tag}: {iters}次 {ms:.2f}ms | 命中{len(rows)}行 | 计划: {plan[0].strip()}")
            return ms, len(rows)

    ms_pre, n_pre = timed("索引前(无idx_dev_ts)")
    with e.begin() as c:
        c.execute(text("CREATE INDEX IF NOT EXISTS idx_dev_ts ON records(point_id, ts)"))
    ms_post, n_post = timed("索引后(有idx_dev_ts)")
    assert n_pre == n_post
    print(f"[p55] 索引收益: {ms_pre:.2f}ms → {ms_post:.2f}ms, 提速 {ms_pre / ms_post:.1f}x")
    print("[p55] 行数与一致性断言全部通过 ✓ (A层窗口函数双实现见 window_queries)")


p55()

## 运行与验证

上方参考实现可**直接运行**（合成数据、纯CPU离线、固定 `SEED` 结果可复现）。
教师可对照学生版逐格讲解：学生版此处留空 → 本notebook给出完成态并附真实输出。

**课堂运行顺序**：cell2(切目录) → Starter 1~9(逐格讲完成态) → A层拓展格(双实现) → `p55()` 自检。
`init_and_seed()` 走 `drop_all → create_all`，整本notebook可反复重跑，输出数字不变。

> 多解法对比见同目录《多解法手册.md》与 `教师参考/solutions_multi*.py`。